# 02 — Experiments: 72-Run Simulation
**Project:** Optimizer and Learning Rate Study of Ovarian Cancer Prediction  
**Course:** HI 192 — Knowledge Representation and Health Decision Support

This is the master experiment notebook. It trains all **72 model configurations** (4 architectures × 6 optimizers × 3 learning rates) and saves every artifact to `results/`.

---
> ## ⚠️ Pre-Computed Results Notice
>
> **Training was executed via `src/run_all.py` as a background process — not from this notebook.**
>
> - All 72 model configurations were trained sequentially by running `python src/run_all.py` directly.
> - Per-run artifacts (weights, training logs, curves, confusion matrices, metrics CSVs) are saved under `results/`.
> - The consolidated results table is at `results/summary/master_results.csv`.
>
> **This notebook does not re-run training.** It serves two purposes:
> 1. **Section 2** — Displays the exact pipeline configuration and experiment matrix that `run_all.py` used.
> 2. **Section 3** — Loads and displays the pre-computed master results from `results/summary/master_results.csv`.
>
> The configuration shown in Section 2 reflects exactly what was executed during training. Do not modify
> these values without re-running `src/run_all.py`, as any change would invalidate the stored results.

---
## Section 0 — Imports and Configuration

In [ ]:
import os
import sys
import pathlib
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Add project root to path so src/ modules are importable
PROJECT_ROOT = pathlib.Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.model_builder import build_model
from src.train import train_model
from src.evaluate import evaluate_model
from src.visualize import plot_accuracy_loss, plot_auc_roc

# ── Global random seed ────────────────────────────────────────────────────────
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# ── Paths ──────────────────────────────────────────────────────────────────────
PROCESSED_DIR = PROJECT_ROOT / 'dataset' / 'processed'
RESULTS_DIR   = PROJECT_ROOT / 'results'

METRICS_DIR      = RESULTS_DIR / 'metrics'
CURVES_ACC_DIR   = RESULTS_DIR / 'curves' / 'accuracy_loss'
CURVES_ROC_DIR   = RESULTS_DIR / 'curves' / 'auc_roc'
CM_DIR           = RESULTS_DIR / 'confusion_matrices'
SUMMARY_DIR      = RESULTS_DIR / 'summary'

for d in [METRICS_DIR, CURVES_ACC_DIR, CURVES_ROC_DIR, CM_DIR, SUMMARY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Experiment matrix ─────────────────────────────────────────────────────────
ARCHITECTURES  = ['VGG19', 'EfficientNetB3', 'ResNet50', 'DenseNet121']
OPTIMIZERS     = ['Adam', 'Adagrad', 'Adamax', 'AdaDelta', 'SGD', 'RMSProp']
LEARNING_RATES = [1e-4, 1e-5, 1e-6]

# ── Fixed training hyperparameters ────────────────────────────────────────────
EPOCHS     = 50
BATCH_SIZE = 32
IMG_SIZE   = (256, 256)

total_runs = len(ARCHITECTURES) * len(OPTIMIZERS) * len(LEARNING_RATES)
print(f"Experiment matrix: {len(ARCHITECTURES)} arch × {len(OPTIMIZERS)} opt × {len(LEARNING_RATES)} LR = {total_runs} runs")

---
## Section 1 — Shared Data Pipeline

> ⚠️ **EXPERIMENTAL INTEGRITY NOTE:** The data split and generators defined in this section are constructed **once** and shared across all 72 simulation runs. Do **not** re-initialize generators inside the training loop. Re-initializing would reshuffle the training set differently per run, introducing data variability as a confound. All observed performance differences must be attributable solely to the choice of optimizer and learning rate.

In [ ]:
# ── Generator definitions (constructed ONCE here, reused across all 84 runs) ──

train_datagen = ImageDataGenerator(
    samplewise_center=True,
    samplewise_std_normalization=True,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.3,
    zoom_range=0.30,
    horizontal_flip=True,
    fill_mode='nearest',
)

eval_datagen = ImageDataGenerator(
    samplewise_center=True,
    samplewise_std_normalization=True,
)

train_generator = train_datagen.flow_from_directory(
    PROCESSED_DIR / 'train',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    seed=RANDOM_SEED,
)

val_generator = eval_datagen.flow_from_directory(
    PROCESSED_DIR / 'val',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False,
)

test_generator = eval_datagen.flow_from_directory(
    PROCESSED_DIR / 'test',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False,
)

CLASS_NAMES = list(train_generator.class_indices.keys())
print(f"Train : {train_generator.samples} images")
print(f"Val   : {val_generator.samples} images")
print(f"Test  : {test_generator.samples} images")
print(f"Classes: {train_generator.class_indices}")

---
## Section 2 — Pipeline Configuration Reference

Training was run via `src/run_all.py`. This section displays the exact pipeline configuration
and experiment matrix that was used during training — for documentation and reproducibility.
No model training is performed here.

**Run naming convention:** `{Architecture}_{Optimizer}_LR{lr}`  
e.g., `ResNet50_Adam_LR1e-4`

In [ ]:
import itertools

# ── Display the full experiment matrix ────────────────────────────────────────
print("=" * 60)
print("EXPERIMENT MATRIX (as executed by src/run_all.py)")
print("=" * 60)
print(f"  Architectures  : {ARCHITECTURES}")
print(f"  Optimizers     : {OPTIMIZERS}")
print(f"  Learning rates : {LEARNING_RATES}")
print(f"  Total runs     : {total_runs}")
print()

# ── Display fixed hyperparameters ─────────────────────────────────────────────
print("FIXED HYPERPARAMETERS")
print("-" * 40)
print(f"  Random seed    : {RANDOM_SEED}")
print(f"  Epochs (max)   : {EPOCHS}")
print(f"  Batch size     : {BATCH_SIZE}")
print(f"  Input size     : {IMG_SIZE}")
print(f"  Dropout        : 0.6 / 0.4 / 0.3  (Dense layers 256/128/64)")
print(f"  Loss           : binary_crossentropy")
print(f"  Monitor metric : val_auc  (EarlyStopping, patience=10)")
print()

# ── Display training augmentation settings ────────────────────────────────────
print("TRAINING AUGMENTATION (train generator only)")
print("-" * 40)
aug_params = {
    "samplewise_center":            True,
    "samplewise_std_normalization": True,
    "rotation_range":               30,
    "width_shift_range":            0.15,
    "height_shift_range":           0.15,
    "shear_range":                  0.3,
    "zoom_range":                   0.30,
    "horizontal_flip":              True,
    "fill_mode":                    "nearest",
}
for k, v in aug_params.items():
    print(f"  {k:<36}: {v}")
print()

# ── List all 72 run names in order ────────────────────────────────────────────
print("ALL 72 RUN NAMES (execution order)")
print("-" * 40)
run_names = [
    f"{arch}_{opt}_LR{lr:.0e}"
    for arch, opt, lr in itertools.product(ARCHITECTURES, OPTIMIZERS, LEARNING_RATES)
]
for i, name in enumerate(run_names, start=1):
    print(f"  {i:02d}. {name}")

---
## Section 3 — Aggregate Results

Load all per-run CSVs from `results/metrics/`, compile into one master DataFrame, and save as `results/summary/all_runs_summary.csv`.

In [ ]:
MASTER_CSV = SUMMARY_DIR / 'master_results.csv'

if not MASTER_CSV.exists():
    raise FileNotFoundError(
        f"Master results file not found: {MASTER_CSV}\n"
        "Run 'python src/run_all.py' from the project root to generate it."
    )

master_df = pd.read_csv(MASTER_CSV)

print(f"Master results loaded from: {MASTER_CSV}")
print(f"Total rows: {len(master_df)}  (expected 72)")
print()

pd.set_option('display.max_rows', 90)
pd.set_option('display.float_format', '{:.4f}'.format)

display(master_df.sort_values('auc', ascending=False).reset_index(drop=True))